# robot trajectory dataset

In [1]:
import h5py

f = h5py.File(
    "../datasets/pickcube/random_episode_000.h5",
    "r"
)

print(list(f.keys()))

['actions', 'observations', 'rewards']


In [2]:
t = 0

print("Observation:")
print(f["observations"][t])

print("\nAction:")
print(f["actions"][t])

print("\nReward:")
print(f["rewards"][t])

Observation:
[[ 3.5281047e-02  4.0070224e-01  1.9574760e-02 -1.9186776e+00
   3.7351161e-02  2.3366489e+00  8.0439991e-01  3.9999999e-02
   3.9999999e-02  0.0000000e+00  0.0000000e+00  0.0000000e+00
   0.0000000e+00  0.0000000e+00  0.0000000e+00  0.0000000e+00
   0.0000000e+00  0.0000000e+00  0.0000000e+00  1.2253533e-02
   3.8011339e-02  1.8215224e-01 -1.7688958e-02  9.9980247e-01
   4.2879577e-03  7.9842266e-03  2.6815735e-02 -1.9813180e-03
   2.8893346e-01 -7.4867904e-04  5.3644367e-02  2.0000000e-02
   5.6876123e-01  0.0000000e+00  0.0000000e+00  8.2250267e-01
  -1.3002212e-02  1.5633028e-02 -1.6215225e-01  2.7564414e-02
  -5.5625685e-02  2.6893345e-01]]

Action:
[ 0.17911236  0.9837746   0.1853819  -0.830926   -0.28162387 -0.42015406
 -0.8190485   0.00515126]

Reward:
[0.07952154]


In [3]:
for key in f.keys():
    print(key, f[key].shape, f[key].dtype)

actions (50, 8) float32
observations (50, 1, 42) float32
rewards (50, 1) float32


In [ ]:
## 3.2 Understand timestamp

In [4]:
T = f["actions"].shape[0]

print("Trajectory length:", T)

Trajectory length: 50


In [5]:
t = 0

obs_t = f["observations"][t]

action_t = f["actions"][t]

reward_t = f["rewards"][t]


print("obs_t shape:", obs_t.shape)
print("action_t:", action_t)
print("reward_t:", reward_t)

obs_t shape: (1, 42)
action_t: [ 0.17911236  0.9837746   0.1853819  -0.830926   -0.28162387 -0.42015406
 -0.8190485   0.00515126]
reward_t: [0.07952154]


## create a new standard version of Robot episod

In [6]:
import h5py
import numpy as np


src = "../datasets/pickcube/random_episode_000.h5"

dst = "../datasets/pickcube/random_episode_standard.h5"


with h5py.File(src, "r") as f:

    observations = f["observations"][:]
    actions = f["actions"][:]
    rewards = f["rewards"][:]


# remove simulation batch dimension
observations = np.squeeze(
    observations,
    axis=1
)


T = actions.shape[0]


timestamps = np.arange(T) * 0.02


with h5py.File(dst, "w") as f:

    f.create_dataset(
        "observations",
        data=observations
    )

    f.create_dataset(
        "actions",
        data=actions
    )

    f.create_dataset(
        "rewards",
        data=rewards
    )

    f.create_dataset(
        "timestamps",
        data=timestamps
    )


    metadata = f.create_group(
        "metadata"
    )

    metadata.attrs["task"] = "PickCube-v1"
    metadata.attrs["robot"] = "Panda"
    metadata.attrs["source"] = "ManiSkill"
    metadata.attrs["control_mode"] = "pd_joint_delta_pos"


print("Saved:", dst)

Saved: ../datasets/pickcube/random_episode_standard.h5


In [7]:
id="5a1f9a"
with h5py.File(dst,"r") as f:

    print(list(f.keys()))

    print(
        dict(
            f["metadata"].attrs
        )
    )

['actions', 'metadata', 'observations', 'rewards', 'timestamps']
{'control_mode': 'pd_joint_delta_pos', 'robot': 'Panda', 'source': 'ManiSkill', 'task': 'PickCube-v1'}


In [8]:
import h5py

path = "../datasets/pickcube/random_episode_standard.h5"

with h5py.File(path,"r") as f:

    for key in f.keys():

        print(key)

        if key != "metadata":
            print(" shape:", f[key].shape)
            print(" dtype:", f[key].dtype)

actions
 shape: (50, 8)
 dtype: float32
metadata
observations
 shape: (50, 42)
 dtype: float32
rewards
 shape: (50, 1)
 dtype: float32
timestamps
 shape: (50,)
 dtype: float64


In [12]:
import h5py

path = "../datasets/pickcube/random_episode_standard.h5"

f = h5py.File(path, "r")

obs = f["observations"][0]
action = f["actions"][0]

print(obs.shape)
print(action.shape)

f["observations"].shape
f["actions"].shape
obs
action

(42,)
(8,)


array([ 0.17911236,  0.9837746 ,  0.1853819 , -0.830926  , -0.28162387,
       -0.42015406, -0.8190485 ,  0.00515126], dtype=float32)

## now we want to check Observation Space

In [14]:
import gymnasium as gym
import mani_skill.envs


env = gym.make(
    "PickCube-v1",
    num_envs=1,
    obs_mode="state",
    control_mode="pd_joint_delta_pos",
)


print(env.observation_space)

obs, info = env.reset(seed=0)

print(type(obs))

Box(-inf, inf, (1, 42), float32)
<class 'torch.Tensor'>


In [15]:
obs, info = env.reset(seed=0)

print(obs.shape)
print(obs)

print(env.observation_space)

torch.Size([1, 42])
tensor([[ 3.5281e-02,  4.0070e-01,  1.9575e-02, -1.9187e+00,  3.7351e-02,
          2.3366e+00,  8.0440e-01,  4.0000e-02,  4.0000e-02,  0.0000e+00,
          0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
          0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  1.2254e-02,
          3.8011e-02,  1.8215e-01, -1.7689e-02,  9.9980e-01,  4.2880e-03,
          7.9842e-03,  2.6816e-02, -1.9813e-03,  2.8893e-01, -7.4868e-04,
          5.3644e-02,  2.0000e-02,  5.6876e-01,  0.0000e+00,  0.0000e+00,
          8.2250e-01, -1.3002e-02,  1.5633e-02, -1.6215e-01,  2.7564e-02,
         -5.5626e-02,  2.6893e-01]])
Box(-inf, inf, (1, 42), float32)


In [17]:
print(env.observation_space)
print(env.unwrapped.observation_space)
print(env)

Box(-inf, inf, (1, 42), float32)
Box(-inf, inf, (1, 42), float32)
<TimeLimitWrapper<OrderEnforcing<PickCubeEnv<PickCube-v1>>>>


In [19]:
base_env = env.unwrapped

print(type(base_env))

<class 'mani_skill.envs.tasks.tabletop.pick_cube.PickCubeEnv'>


In [20]:
print(base_env.observation_space)

Box(-inf, inf, (1, 42), float32)


In [22]:
print(base_env.agent)
print(dir(base_env.agent))

['__annotations__', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_after_init', '_after_loading_articulation', '_agent_idx', '_control_freq', '_control_mode', '_controller_configs', '_default_control_mode', '_load_articulation', '_sensor_configs', 'action_space', 'arm_damping', 'arm_force_limit', 'arm_joint_names', 'arm_stiffness', 'before_simulation_step', 'build_grasp_pose', 'build_separate', 'control_mode', 'controller', 'controllers', 'device', 'disable_self_collisions', 'ee_link_name', 'finger1_link', 'finger1pad_link', 'finger2_link', 'finger2pad_link', 'fix_root_link', 'get_controller_state', 'get_proprioception', 'get_state', 'gripper_damping', 'gripper_force_limit', '

In [23]:
print(base_env.agent.action_space)

Box(-1.0, 1.0, (8,), float32)


In [24]:
proprio = base_env.agent.get_proprioception()

print(type(proprio))
print(proprio)

<class 'dict'>
{'qpos': tensor([[ 0.0353,  0.4007,  0.0196, -1.9187,  0.0374,  2.3366,  0.8044,  0.0400,
          0.0400]]), 'qvel': tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0.]])}


In [25]:
state = base_env.agent.get_state()

print(type(state))
print(state)

<class 'dict'>
{'robot_root_pose': Pose(raw_pose=tensor([[-6.1500e-01,  7.2760e-11, -1.4901e-08,  1.0000e+00,  0.0000e+00,
          0.0000e+00,  0.0000e+00]])), 'robot_root_vel': tensor([[0., 0., 0.]]), 'robot_root_qvel': tensor([[0., 0., 0.]]), 'robot_qpos': tensor([[ 0.0353,  0.4007,  0.0196, -1.9187,  0.0374,  2.3366,  0.8044,  0.0400,
          0.0400]]), 'robot_qvel': tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0.]]), 'controller': {}}


In [26]:
print(base_env.agent.tcp_pose)
print(base_env.agent.tcp_pos)

Pose(raw_pose=tensor([[ 0.0123,  0.0380,  0.1822, -0.0177,  0.9998,  0.0043,  0.0080]]))
tensor([[0.0123, 0.0380, 0.1822]])


In [28]:
obs, info = env.reset(seed=0)

print([x for x in dir(base_env) if "obs" in x.lower()])

['SUPPORTED_OBS_MODES', '_flatten_raw_obs', '_get_obs_agent', '_get_obs_extra', '_get_obs_sensor_data', '_get_obs_state_dict', '_get_obs_with_sensor_data', '_init_raw_obs', '_last_obs', '_obs_mode', 'get_obs', 'obs_mode', 'obs_mode_struct', 'observation_space', 'single_observation_space', 'update_obs_space']


['SUPPORTED_OBS_MODES', '_flatten_raw_obs', '_get_obs_agent', '_get_obs_extra', '_get_obs_sensor_data', '_get_obs_state_dict', '_get_obs_with_sensor_data', '_init_raw_obs', '_last_obs', '_obs_mode', 'get_obs', 'obs_mode', 'obs_mode_struct', 'observation_space', 'single_observation_space', 'update_obs_space']

obs=flatten(agent_state+extra_state)

In [32]:
obs, info = env.reset(seed=0)

raw_obs = base_env._get_obs_state_dict(info)

print(type(raw_obs))

print(raw_obs.keys())

<class 'dict'>
dict_keys(['agent', 'extra'])


In [34]:
print(raw_obs["agent"].keys())

dict_keys(['qpos', 'qvel'])


In [35]:
for k, v in raw_obs["agent"].items():
    print("================")
    print(k)
    print(type(v))
    if hasattr(v, "shape"):
        print("shape:", v.shape)
    print(v)

qpos
<class 'torch.Tensor'>
shape: torch.Size([1, 9])
tensor([[ 0.0353,  0.4007,  0.0196, -1.9187,  0.0374,  2.3366,  0.8044,  0.0400,
          0.0400]])
qvel
<class 'torch.Tensor'>
shape: torch.Size([1, 9])
tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0.]])


In [36]:
print(raw_obs["extra"].keys())

for k, v in raw_obs["extra"].items():
    print("================")
    print(k)
    print(type(v))
    if hasattr(v, "shape"):
        print("shape:", v.shape)
    print(v)

dict_keys(['is_grasped', 'tcp_pose', 'goal_pos', 'obj_pose', 'tcp_to_obj_pos', 'obj_to_goal_pos'])
is_grasped
<class 'torch.Tensor'>
shape: torch.Size([1])
tensor([False])
tcp_pose
<class 'torch.Tensor'>
shape: torch.Size([1, 7])
tensor([[ 0.0123,  0.0380,  0.1822, -0.0177,  0.9998,  0.0043,  0.0080]])
goal_pos
<class 'torch.Tensor'>
shape: torch.Size([1, 3])
tensor([[ 0.0268, -0.0020,  0.2889]])
obj_pose
<class 'torch.Tensor'>
shape: torch.Size([1, 7])
tensor([[-7.4868e-04,  5.3644e-02,  2.0000e-02,  5.6876e-01,  0.0000e+00,
          0.0000e+00,  8.2250e-01]])
tcp_to_obj_pos
<class 'torch.Tensor'>
shape: torch.Size([1, 3])
tensor([[-0.0130,  0.0156, -0.1622]])
obj_to_goal_pos
<class 'torch.Tensor'>
shape: torch.Size([1, 3])
tensor([[ 0.0276, -0.0556,  0.2689]])


# PickCube Robot Observation Schema

## Overview

In ManiSkill, the raw observation returned by the environment is a flattened tensor:

\[
o_t \in R^{42}
\]

Although the observation appears as a 42-dimensional vector, it is composed of multiple semantic components:

\[
Observation =
Agent\ State + Extra\ Task\ State
\]

The structured observation before flattening is:

```text
Observation

├── agent
│   ├── robot_qpos
│   └── robot_qvel
│
└── extra
    ├── tcp_pose
    ├── goal_pos
    ├── obj_pose
    ├── tcp_to_obj_pos
    ├── obj_to_goal_pos
    └── is_grasped

In [37]:
print(base_env.agent.arm_joint_names)
print(base_env.agent.gripper_joint_names)

['panda_joint1', 'panda_joint2', 'panda_joint3', 'panda_joint4', 'panda_joint5', 'panda_joint6', 'panda_joint7']
['panda_finger_joint1', 'panda_finger_joint2']
